<a href="https://colab.research.google.com/github/MtHenriqueF/data-pipeline-spark-streamlit/blob/main/Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Montando Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os

WORKDIR = '/content/drive/MyDrive/data-science/Big data - aula/'

os.chdir(WORKDIR)
print('Diretório atual:', os.getcwd())


Diretório atual: /content/drive/MyDrive/data-science/Big data - aula


# Baixando dependências

In [ ]:
!pip install pyspark

In [ ]:
!pip install streamlit

In [ ]:
!pip install pyngrok --quiet

# Pyspark

## 1 Carga e pré-processamento (PySpark)

### a) Ler o CSV usando PySpark

In [ ]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Exercicio3").getOrCreate()
df = spark.read.csv("super_mario_dataset.csv"
, header=True, inferSchema=True)

### b) Inspecionar o dataset

In [ ]:
df.show()

+---------+-----+-----+----------+---------------+-------------+------------+----------------+------+----------+
|player_id|world|level|lives_left|coins_collected|powerups_used|time_seconds|enemies_defeated| score|      date|
+---------+-----+-----+----------+---------------+-------------+------------+----------------+------+----------+
| player_7|    7|    1|       4.0|           60.0|          1.0|       305.0|            32.0|9786.0|2025-09-16|
| player_4|    1|    1|       1.0|           37.0|          2.0|       418.0|            18.0|6372.0|2025-09-24|
|player_13|    1|    4|      NULL|            2.0|          4.0|       473.0|            38.0|9818.0|2025-10-09|
|player_15|    3|    3|       3.0|           83.0|         NULL|       456.0|            23.0|5985.0|2025-10-02|
|player_11|    7|    1|       1.0|           69.0|         NULL|        86.0|            17.0|7562.0|2025-09-27|
| player_8|    8|    2|      NULL|           NULL|          0.0|       595.0|            34.0|59

In [ ]:
df.describe()

DataFrame[summary: string, player_id: string, world: string, level: string, lives_left: string, coins_collected: string, powerups_used: string, time_seconds: string, enemies_defeated: string, score: string]

In [ ]:
from pyspark.sql.functions import col, count, when, isnan
def contagem_nulos(df):
  tipos_numericos = ['float', 'double']

  expressoes_contagem = []

  for coluna, dtype in df.dtypes:
      if dtype in tipos_numericos:

          expressoes_contagem.append(
              count(when(isnan(coluna) | col(coluna).isNull(), coluna)).alias(coluna)
          )
      else:
          expressoes_contagem.append(
              count(when(col(coluna).isNull(), coluna)).alias(coluna)
          )

  return df.select(expressoes_contagem).show()

In [ ]:
from pyspark.ml.feature import Imputer

imputer = Imputer(
    inputCols=["lives_left", "coins_collected", "powerups_used", "time_seconds", "enemies_defeated"],
    outputCols=["lives_left", "coins_collected", "powerups_used", "time_seconds", "enemies_defeated"]
).setStrategy("mean")



In [ ]:
model = imputer.fit(df)
df_imputed = model.transform(df)


In [ ]:
df_imputed.show()

+---------+-----+-----+------------------+-----------------+------------------+------------+------------------+------+----------+
|player_id|world|level|        lives_left|  coins_collected|     powerups_used|time_seconds|  enemies_defeated| score|      date|
+---------+-----+-----+------------------+-----------------+------------------+------------+------------------+------+----------+
| player_7|    7|    1|               4.0|             60.0|               1.0|       305.0|              32.0|9786.0|2025-09-16|
| player_4|    1|    1|               1.0|             37.0|               2.0|       418.0|              18.0|6372.0|2025-09-24|
|player_13|    1|    4|2.4549071618037135|              2.0|               4.0|       473.0|              38.0|9818.0|2025-10-09|
|player_15|    3|    3|               3.0|             83.0|2.5532105972159855|       456.0|              23.0|5985.0|2025-10-02|
|player_11|    7|    1|               1.0|             69.0|2.5532105972159855|        86.

In [ ]:
from pyspark.sql.functions import round

df_imputed = df_imputed.withColumn("lives_left", round(col("lives_left"), 2)) \
                       .withColumn("coins_collected", round(col("coins_collected"), 2)) \
                       .withColumn("powerups_used", round(col("powerups_used"))) \
                       .withColumn("time_seconds", round(col("time_seconds"), 2)) \
                       .withColumn("enemies_defeated", round(col("enemies_defeated"), 2))


In [ ]:
df_imputed.show()

+---------+-----+-----+----------+---------------+-------------+------------+----------------+------+----------+
|player_id|world|level|lives_left|coins_collected|powerups_used|time_seconds|enemies_defeated| score|      date|
+---------+-----+-----+----------+---------------+-------------+------------+----------------+------+----------+
| player_7|    7|    1|       4.0|           60.0|          1.0|       305.0|            32.0|9786.0|2025-09-16|
| player_4|    1|    1|       1.0|           37.0|          2.0|       418.0|            18.0|6372.0|2025-09-24|
|player_13|    1|    4|      2.45|            2.0|          4.0|       473.0|            38.0|9818.0|2025-10-09|
|player_15|    3|    3|       3.0|           83.0|          3.0|       456.0|            23.0|5985.0|2025-10-02|
|player_11|    7|    1|       1.0|           69.0|          3.0|        86.0|            17.0|7562.0|2025-09-27|
| player_8|    8|    2|      2.45|          50.79|          0.0|       595.0|            34.0|59

In [ ]:
contagem_nulos(df_imputed)

+---------+-----+-----+----------+---------------+-------------+------------+----------------+-----+----+
|player_id|world|level|lives_left|coins_collected|powerups_used|time_seconds|enemies_defeated|score|date|
+---------+-----+-----+----------+---------------+-------------+------------+----------------+-----+----+
|        0|    0|    0|         0|              0|            0|           0|               0|    0|   0|
+---------+-----+-----+----------+---------------+-------------+------------+----------------+-----+----+



In [ ]:
df_imputed.toPandas().to_parquet('imputed_data.parquet')

##2 - Sumarização e agregação e 3 - Salvando como parquet e transformando para pandas.

### a. Pontuação média por mundo.

In [ ]:
from pyspark.sql.functions import avg
df_score_per_world = df_imputed.groupby('world').agg(avg(col("score")).alias("score_per_world"))

In [ ]:
df_score_per_world.show()

+-----+-----------------+
|world|  score_per_world|
+-----+-----------------+
|    1|5051.817948717949|
|    6|4973.518518518518|
|    3|5035.151193633952|
|    5|4774.147668393783|
|    4|4933.936430317848|
|    8|4844.303278688524|
|    7|4846.551724137931|
|    2|5218.846820809248|
+-----+-----------------+



In [ ]:
# df_score_per_world.toPandas().to_parquet('score_per_world.parquet')

###b. Tempo médio de conclusão por mundo e nível.

In [ ]:
df_avgTime_per_world_level = df_imputed.groupBy('world', 'level').agg(round(avg(col("time_seconds")), 2).alias("time_per_world_level"))

In [ ]:
df_avgTime_per_world_level.show()

+-----+-----+--------------------+
|world|level|time_per_world_level|
+-----+-----+--------------------+
|    6|    1|               322.6|
|    3|    1|              345.09|
|    7|    4|              332.71|
|    2|    2|              329.95|
|    8|    3|              295.43|
|    7|    1|              283.58|
|    2|    3|              308.98|
|    1|    2|               303.6|
|    1|    1|              322.84|
|    1|    3|               320.6|
|    7|    2|              318.79|
|    1|    4|              326.33|
|    5|    4|              313.89|
|    7|    3|              333.45|
|    3|    3|              307.87|
|    8|    1|              314.36|
|    4|    3|              291.88|
|    2|    1|              325.02|
|    2|    4|              310.88|
|    6|    3|              311.78|
+-----+-----+--------------------+
only showing top 20 rows



In [ ]:
# df_avgTime_per_world_level.toPandas().to_parquet('avgTime_per_world_level.parquet')

###c. Top 5 jogadores com maior pontuação média.

In [ ]:
df_top_5_player = df_imputed.groupby('player_id').agg(avg(col('score')).alias('top_5_players')).limit(5)

In [ ]:
df_top_5_player.show()

+---------+------------------+
|player_id|     top_5_players|
+---------+------------------+
| player_7| 4723.901554404145|
|player_14| 4864.030150753769|
| player_5|4710.9567307692305|
| player_8|          5277.045|
|player_10| 5138.023923444976|
+---------+------------------+



In [ ]:
# df_top_5_player.toPandas().to_parquet('top_5_player.parquet')

###d. Contagem de partidas por data.

In [ ]:
df_match_count = df_imputed.groupBy("date").count().alias('match_count')

In [ ]:
df_match_count.show()

+----------+-----+
|      date|count|
+----------+-----+
|2025-09-23|   89|
|2025-09-25|   95|
|2025-09-10|  100|
|2025-09-11|   84|
|2025-09-16|  112|
|2025-09-18|  103|
|2025-09-13|  105|
|2025-09-09|  105|
|2025-10-01|  112|
|2025-09-27|   78|
|2025-10-04|  102|
|2025-09-21|   98|
|2025-10-03|   89|
|2025-09-24|   96|
|2025-10-07|   96|
|2025-10-06|  112|
|2025-09-15|  104|
|2025-09-14|  102|
|2025-10-09|   87|
|2025-09-26|   93|
+----------+-----+
only showing top 20 rows



In [ ]:
# df_match_count.toPandas().to_parquet('match_count.parquet')

# Fechando sessao spark

In [ ]:
# spark.stop()

# 4-Pandas, matplotlib e streamlit

##

###a. Carregue o arquivo .parquet com pandas.

In [ ]:
import pandas as pd
df_score_per_world = pd.read_parquet('score_per_world.parquet')

In [ ]:
df_avgTime_per_world_level = pd.read_parquet('avgTime_per_world_level.parquet')

In [ ]:
df_top_5_player = pd.read_parquet('top_5_player.parquet')

In [ ]:
df_match_count = pd.read_parquet('match_count.parquet')

In [ ]:
worlds = sorted(df_imputed.select("world").distinct().toPandas()["world"].tolist())
players = sorted(df_imputed.select("player_id").distinct().toPandas()["player_id"].tolist())

In [ ]:
import json

worlds = sorted(df_imputed.select("world").distinct().toPandas()["world"].tolist())
players = sorted(df_imputed.select("player_id").distinct().toPandas()["player_id"].tolist())

with open("worlds.json", "w") as f:
    json.dump(worlds, f)

with open("players.json", "w") as f:
    json.dump(players, f)


In [ ]:
app_code = """
import streamlit as st
import pandas as pd
import matplotlib.pyplot as plt

st.title("🎮 Dashboard de Desempenho - Mario Game")

@st.cache_data
def load_data():
    return pd.read_parquet("imputed_data.parquet")

df = load_data()

worlds = sorted(df["world"].dropna().unique().tolist())
players = sorted(df["player_id"].dropna().unique().tolist())

selected_world = st.multiselect("🌍 Selecione o(s) Mundo(s):", worlds)
selected_player = st.multiselect("👤 Selecione o(s) Jogador(es):", players)

df_filtered = df.copy()

if selected_world:
    df_filtered = df_filtered[df_filtered["world"].isin(selected_world)]

if selected_player:
    df_filtered = df_filtered[df_filtered["player_id"].isin(selected_player)]


# i. Pontuação média por mundo
df_score_per_world = (
    df_filtered.groupby("world", as_index=False)["score"]
    .mean()
    .rename(columns={"score": "score_per_world"})
)

# ii. Número de partidas por data (DINÂMICO)
df_match_count = (
    df_filtered.groupby("date", as_index=False)
    .size()
    .rename(columns={"size": "match_count"})
    .sort_values("date")
)

# iii. Distribuição de moedas coletadas
coins = df_filtered["coins_collected"].dropna()


st.subheader("🏆 Pontuação média por mundo")
if not df_score_per_world.empty:
    fig1, ax1 = plt.subplots()
    ax1.bar(df_score_per_world["world"], df_score_per_world["score_per_world"], color="skyblue")
    ax1.set_xlabel("Mundo")
    ax1.set_ylabel("Pontuação média")
    st.pyplot(fig1)
else:
    st.warning("Nenhum dado disponível para o filtro selecionado.")

st.subheader("Número de partidas por data")
if not df_match_count.empty:
    fig2, ax2 = plt.subplots()
    ax2.plot(df_match_count["date"], df_match_count["match_count"], marker="o", color="orange")
    ax2.set_xlabel("Data")
    ax2.set_ylabel("Número de partidas")
    plt.xticks(rotation=45)
    st.pyplot(fig2)
else:
    st.warning("Nenhum dado de partidas para o filtro selecionado.")

st.subheader("Distribuição de moedas coletadas")
if not coins.empty:
    fig3, ax3 = plt.subplots()
    ax3.hist(coins, bins=20, color="green", alpha=0.7)
    ax3.set_xlabel("Moedas coletadas")
    ax3.set_ylabel("Frequência")
    st.pyplot(fig3)
else:
    st.warning("Nenhum dado de moedas para o filtro selecionado.")


"""

with open("app.py", "w") as f:
    f.write(app_code)

In [ ]:

import os
os.environ["NGROK_AUTHTOKEN"] = "33tC6DQxuwWOTO7j9nBvhtzSEAZ_6b3zFJtLyR1UVkyhWCys2"
from pyngrok import ngrok
tunnel = ngrok.connect(8501, "http")
print(tunnel)

NgrokTunnel: "https://unwarpable-ambroise-blissfully.ngrok-free.dev" -> "http://localhost:8501"


In [ ]:
!streamlit run app.py &


  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.125.158.45:8501

  Stopping...


In [ ]:
!pip install streamlit